# ClinKriya GRPO Training with Unsloth + TRL

Fine-tune **Qwen3-1.7B** on the clinkriya clinical decision-making benchmark using:
- **Unsloth** for 2-4x faster LoRA fine-tuning
- **TRL GRPOTrainer** with environment `tool-use` rollouts
- **Mock FHIR** backend — no live server, runs fully offline
- **Named tool calls** matching benchmark evaluation format

**Runtime:** T4 minimum, A100/H100 recommended

## 1. Install Dependencies & Clone Repo

In [1]:
# Install training stack
!pip install -q unsloth trl datasets huggingface_hub

# Clone repo (contains medagentbench_env + benchmark data)
!git clone https://github.com/ananya173147/clinKriya.git ./repo

# Verify GPU
!nvidia-smi | head -20

fatal: destination path './repo' already exists and is not an empty directory.
Sun Mar  8 18:43:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   26C    P0             69W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                  

In [2]:
pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo


Note: you may need to restart the kernel to use updated packages.


In [3]:
!wget -O trainer.ipynb "https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game_BF16.ipynb"

--2026-03-08 18:43:53--  https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game_BF16.ipynb
Resolving colab.research.google.com (colab.research.google.com)... 142.250.189.174, 2001:4860:4802:32::180, 2001:4860:4802:36::180, ...
Connecting to colab.research.google.com (colab.research.google.com)|142.250.189.174|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘trainer.ipynb’

trainer.ipynb           [ <=>                ]  92.21K  --.-KB/s    in 0.06s   

2026-03-08 18:43:53 (1.59 MB/s) - ‘trainer.ipynb’ saved [94422]



In [10]:
!git -C ./repo pull 

/opt/conda/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=8075) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Already up to date.


## 2. Load Model with Unsloth (4-bit LoRA)

In [5]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

print(f"Model: {MODEL_NAME}")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.13/site-packages/triton/runtime/autotuner.py:101: DeprecationWarning: warmup, rep, and use_cuda_graph parameters are deprecated. See https://github.com/triton-lang/triton/pull/4496 for details.
  warnings.warn(("warmup, rep, and use_cuda_graph parameters are deprecated. See "


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/opt/conda/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=8075) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model: Qwen/Qwen3-1.7B
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 3. Load Environment & Data

In [11]:
import importlib.util, sys
from pathlib import Path

REPO = Path("./repo")

# Load train.py directly — avoids installing openenv-core
spec = importlib.util.spec_from_file_location(
    "train", REPO / "medagentbench_env" / "train.py"
)
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

MedAgentTrainEnv = train_mod.MedAgentTrainEnv
reward_func      = train_mod.reward_func
build_dataset    = train_mod.build_dataset

DATA_DIR = REPO / "medagentbench_env" / "data"

# Override module paths to point at our cloned data
train_mod._DATA_DIR   = DATA_DIR
train_mod._CACHE_PATH = DATA_DIR / "fhir_cache.json"
train_mod._SYSTEM_PROMPT_PATH = DATA_DIR / "new_system.txt"

# Pre-load shared resources
train_mod._get_mock_fhir()
tasks = train_mod._get_tasks()
print(f"FHIR cache loaded | {len(tasks)} tasks")
print(f"System prompt: {train_mod._get_system_prompt()}...")

FHIR cache loaded | 90 tasks
System prompt: You are a medical AI agent that completes clinical tasks by interacting with a FHIR EHR. The patient MRN is always provided in the task — do not search for patients.

You have access to these tools:

- fhir_observation_search(patient, code): search lab results for a patient by LOINC or local code
- fhir_vitals_create(resourceType, category, code, effectiveDateTime, status, valueString, subject): record a vital sign observation
- fhir_service_request_create(resourceType, code, authoredOn, status, intent, priority, subject, note): create a referral or lab order
- finish(value): submit your final answer as a list

Rules:
- Only perform an action when its precondition is explicitly met. If the task says "if X then do Y", only do Y when X is true.
- Always call finish to submit your answer. Never respond with an empty list.
- Numbers must be numbers (not strings). Dates must be ISO format strings. Do not include units.

Good finish examples:
- fin

In [12]:
# Sanity check — run one episode end-to-end
env = MedAgentTrainEnv()
instruction = env.reset()
task = env._task
print(f"Task : {task['id']}  ({task.get('_benchmark_type')})")
print(f"Instr: {instruction[:120]}...")

# Simulate a correct BP observation POST
resp = env.fhir_vitals_create(
    resourceType="Observation",
    category=[{"coding": [{"code": "vital-signs"}]}],
    code={"text": "BP"},
    effectiveDateTime="2023-11-13T10:15:00",
    status="final",
    valueString="118/77",
    subject={"reference": f"Patient/{task['eval_MRN']}"},
)
print(f"POST: {resp[:60]}")

result = env.finish([])
print(f"Finish: {result}")

Task : task3_1  (always-action)
Instr: I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context...
POST: POST request accepted and executed successfully. Please call

────────────────────────────────────────────────────────────
EPISODE TRACE  task=task3_1  steps=2  reward=0.425
────────────────────────────────────────────────────────────
  [1] AGENT: POST http://localhost:8080/fhir/Observation
{"resourceType": "Observation", "status": "final", "category": [{"coding": [{"code": "vital-signs"}]}], "code": {"text": "BP"}, "effectiveDateTime": "2023-11-13T10:15:00", "valueString": "118/77", "subject": {"reference": "Patient/S2380121"}}
  [2] ENV  : POST request accepted and executed successfully. Please call finish if you have got answers for all the questions and finished all the requested tasks
  [3] AGENT: FINISH([])
  [4] ENV  : Task completed.
  ANSWER: []
────────────────────────────────────────────────────────────
Finis

## 4. Build Training Dataset

In [13]:
dataset = build_dataset(DATA_DIR)
print(f"Dataset: {len(dataset)} tasks")
print(f"Roles  : {[m['role'] for m in dataset[0]['prompt']]}")
print(f"User   : {dataset[0]['prompt'][1]['content']}")

Dataset: 90 tasks
Roles  : ['system', 'user']
User   : I just measured the blood pressure for patient with MRN of S2380121, and it is "118/77 mmHg". Help me record it.
Context: It's 2023-11-13T10:15:00+00:00 now. The flowsheet ID for blood pressure is BP.


## 5. Train with GRPOTrainer

In [ ]:
import logging
# Suppress malformed deprecation warning from transformers 5.2.0
# (warning_once passes FutureWarning as % arg but message has no %s)
logging.getLogger('transformers.modeling_attn_mask_utils').setLevel(logging.ERROR)

from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./medagent_grpo_output"

# Use only 5 tasks for a minimal smoke-test run
small_dataset = dataset.select(range(5))

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_completion_length=512,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=2,
    logging_steps=1,
    log_completions=True,
    num_completions_to_print=1,
    # fp16=True + bf16=False: Unsloth's LoRA kernel has a hardcoded fp16
    # autocaster; bf16=True causes Half/BFloat16 mismatch at matmul_lora.
    fp16=False,
    bf16=True,
    save_steps=50,
    save_total_limit=2,
    # num_generations=4: default 8 means 8x completions per prompt.
    # With 5 tasks that means 5x8=40 rollouts before first gradient update.
    num_generations=2,
    beta=0.01,
    temperature=0.9,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    train_dataset=small_dataset,
    environment_factory=MedAgentTrainEnv,
    processing_class=tokenizer,
    args=grpo_config,
)

print("Starting GRPO training on 5 tasks...")
trainer.train()

## 6. Save Model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to {OUTPUT_DIR}")

# Merge LoRA into base weights (optional — needed for full model push)
# merged = model.merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + "_merged")
# tokenizer.save_pretrained(OUTPUT_DIR + "_merged")

## 7. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN    = "hf_xxx"          # your HF write token
HF_REPO_ID  = "YOUR_USERNAME/medagent-qwen3-1.7b"

login(token=HF_TOKEN)

# Push LoRA adapter
model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

## 8. Quick Evaluation

Run the trained model on a sample of tasks. The model generates tool calls;
we parse Qwen3's `<tool_call>` format and route to the environment methods.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (REPO / "medagentbench_env" / "data" / "new_system.txt").read_text().strip()


def parse_tool_call(text: str):
    """Parse Qwen3 <tool_call>{...}</tool_call> format."""
    m = re.search(r'<tool_call>\s*({.*?})\s*</tool_call>', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    # Fallback: bare {"name": ..., "arguments": ...}
    m = re.search(r'\{\s*"name"\s*:\s*"(\w+)"\s*,\s*"arguments"\s*:\s*(\{.*?\})\s*\}',
                  text, re.DOTALL)
    if m:
        try:
            return {"name": m.group(1), "arguments": json.loads(m.group(2))}
        except json.JSONDecodeError:
            pass
    return None


def run_episode(env, max_steps=8):
    """Run one episode with the trained model, return reward and trace."""
    instruction = env.reset()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": instruction},
    ]

    for step in range(max_steps):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
        reply = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

        tc = parse_tool_call(reply)
        if tc is None:
            # No parseable tool call — end episode
            env.finish([])
            break

        tool_name = tc.get("name", "")
        tool_args = tc.get("arguments", {})
        method = getattr(env, tool_name, None)

        if method is None or tool_name.startswith("_"):
            env.finish([])
            break

        try:
            tool_result = method(**tool_args)
        except Exception as e:
            tool_result = f"Tool error: {e}"

        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "tool",      "content": tool_result,
                         "tool_call_id": tool_name})

        if env.done:
            break

    return env.reward


# Reset task counter so we evaluate from the start
train_mod._TASK_INDEX = 0

NUM_EVAL = 10
rewards = []

for i in range(NUM_EVAL):
    env = MedAgentTrainEnv()
    r = run_episode(env)
    rewards.append(r)
    print(f"  {env._task['id']}: reward={r:.3f}")

avg = sum(rewards) / len(rewards)
print(f"\nPost-training avg reward ({NUM_EVAL} tasks): {avg:.4f}")
print(f"Baseline (Qwen3-8B OpenRouter):             0.2748")

## 9. Load from HuggingFace (clone)

Re-load the pushed model on any machine — no repo clone needed.

In [ ]:
# from unsloth import FastLanguageModel
# 
# HF_REPO_ID = "YOUR_USERNAME/medagent-qwen3-1.7b"
# 
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=HF_REPO_ID,
#     max_seq_length=4096,
#     load_in_4bit=True,
# )
# FastLanguageModel.for_inference(model)
# print(f"Loaded {HF_REPO_ID} from HuggingFace Hub")